In [13]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("Messy_Employee_dataset.csv")

print("Original Shape:", df.shape)

# ==========================================
# 1. CHECK MISSING VALUES
# ==========================================
print("\nMissing Values:")
print(df.isnull().sum())

# Fill missing numeric values with median
numeric_cols = df.select_dtypes(include=['number']).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill missing categorical values with mode
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# ==========================================
# 2. REMOVE DUPLICATES
# ==========================================
duplicates = df.duplicated().sum()
print(f"\nDuplicate Rows Found: {duplicates}")

df = df.drop_duplicates()

# ==========================================
# 3. CLEAN COLUMN NAMES
# ==========================================
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

# ==========================================
# 4. STANDARDIZE TEXT VALUES
# ==========================================
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()

# Example Gender Standardization
if 'gender' in df.columns:
    df['gender'] = (
        df['gender']
        .str.lower()
        .replace({
            'm': 'male',
            'male': 'male',
            'f': 'female',
            'female': 'female'
        })
    )

# Example Country Standardization
if 'country' in df.columns:
    df['country'] = (
        df['country']
        .str.lower()
        .replace({
            'usa': 'united states',
            'u.s.a': 'united states',
            'us': 'united states',
            'uk': 'united kingdom'
        })
    )

# ==========================================
# 5. DATE FORMAT CONVERSION
# ==========================================
date_columns = ['join_date']

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(
            df[col],
            errors='coerce'
        )

        df[col] = df[col].dt.strftime('%d-%m-%Y')

# ==========================================
# 6. FIX DATA TYPES
# ==========================================

# Age
if 'age' in df.columns:
    df['age'] = pd.to_numeric(
        df['age'],
        errors='coerce'
    )

    df['age'] = df['age'].fillna(
        df['age'].median()
    )

    df['age'] = df['age'].astype(int)

# Salary
if 'salary' in df.columns:
    df['salary'] = (
        df['salary']
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.replace('$', '', regex=False)
    )

    df['salary'] = pd.to_numeric(
        df['salary'],
        errors='coerce'
    )

    df['salary'] = df['salary'].fillna(
        df['salary'].median()
    )

# ==========================================
# 7. REMOVE EXTRA SPACES
# ==========================================
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# ==========================================
# 8. OUTLIER DETECTION (IQR METHOD)
# ==========================================
if 'salary' in df.columns:

    Q1 = df['salary'].quantile(0.25)
    Q3 = df['salary'].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df['salary'] < lower) |
        (df['salary'] > upper)
    ]

    print(f"\nSalary Outliers Found: {len(outliers)}")

# ==========================================
# 9. DATA VALIDATION
# ==========================================
if 'age' in df.columns:
    df = df[
        (df['age'] >= 18) &
        (df['age'] <= 65)
    ]

# ==========================================
# 10. CHECK FINAL DATA TYPES
# ==========================================
print("\nFinal Data Types:")
print(df.dtypes)

print("\nFinal Shape:", df.shape)

# ==========================================
# SAVE CLEANED DATA
# ==========================================
df.to_csv(
    "cleaned_employee_dataset.csv",
    index=False
)

print("\nCleaning Completed!")
print("Saved as: cleaned_employee_dataset.csv")

Original Shape: (1020, 12)

Missing Values:
Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64

Duplicate Rows Found: 0

Salary Outliers Found: 0

Final Data Types:
employee_id           object
first_name            object
last_name             object
age                    int64
department_region     object
status                object
join_date             object
salary               float64
email                 object
phone                  int64
performance_score     object
remote_work             bool
dtype: object

Final Shape: (1020, 12)

Cleaning Completed!
Saved as: cleaned_employee_dataset.csv
